<a href="https://colab.research.google.com/github/ja2faded/ds2002-fa26/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
df_original = df.copy()

df= df.drop_duplicates()
log('drop_duplicates', 'removed exact duplicate rows' , (len(df_original) - len(df)))
print(df)



[drop_duplicates] removed exact duplicate rows (15 row(s))
     order_id           item  category  qty   price
0           0    Rain Poncho  RainGear  3.0  $12.00
1           1    foam finger   Apparel  1.0     7.5
2           2  cheese burger     Merch  1.0   $7.50
3           3   Cheeseburger      Food  NaN   $7.50
4           4  cheese burger   Apparel  1.0     7.5
..        ...            ...       ...  ...     ...
295       295    rain poncho      food  3.0     6.0
296       296  cheese burger  RainGear -1.0     6.0
297       297    Foam Finger      food  1.0     7.5
298       298    Rain Poncho     Merch  1.0   $7.50
299       299  cheese burger      food  1.0  $12.00

[300 rows x 5 columns]


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].str.replace('$','').astype(float)
log('clean price', 'converted to float', len(df))
#does not affect revenue total




[clean price] converted to float (300 row(s))


In [5]:
#since str cannot be multiplied, check revenue change from TODO1

revenue_before_duplicates = (
                            pd.to_numeric(df_original['qty'], errors='coerce')
                            * df_original['price'].str.replace('$', '', regex =False)
                            .astype(float)
                            )


revenue_after_duplicates = (pd.to_numeric(df['qty'], errors = 'coerce') * df['price'])

total_before = revenue_before_duplicates.sum()
print('Total before:' , total_before )
total_after = revenue_after_duplicates.sum()

print('Revenue difference after duplicates removed', (revenue_before_duplicates.sum() - revenue_after_duplicates.sum()))

Total before: 4852.5
Revenue difference after duplicates removed 258.0


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [6]:
#check revenue before
revenue_before_qty = (pd.to_numeric(df['qty'], errors='coerce') * df['price'])

df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
missing_or_negative = df[(df['qty'].isna()) | (df['qty'] < 0)]
df = df.drop(missing_or_negative.index)

#check revenue after
revenue_after_qty = (pd.to_numeric(df['qty'], errors='coerce') * df['price'])

print('Revenue Difference: ', revenue_before_qty.sum() - revenue_after_qty.sum())


log('clean qty', f'dropped {len(missing_or_negative)} rows with missing or negative qty', len(missing_or_negative))

Revenue Difference:  -145.5
[clean qty] dropped 25 rows with missing or negative qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [7]:
#does not affect revenue total
before_item= df['item'].copy()

#what you have before
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger', 'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger','foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho','rain poncho': 'Rain Poncho',
}

df['item'] = df['item'].map(ITEM_MAP)

log('canonicalize_item', 'turned 6 spellings into 3 items via ITEM_MAP', (before_item != df['item']).sum())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[canonicalize_item] turned 6 spellings into 3 items via ITEM_MAP (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [8]:
#does not affect revenue total
before_cat = df['category'].copy()

#what you have before
print(df['category'].value_counts())

CAT_MAP = {
          'Food': 'Food', 'food': 'Food',
          'Merch': 'Merch', 'Apparel': 'Merch',
          'RainGear': 'RainGear', 'rain-gear': 'RainGear'
          }

df['category'] = df['category'].map(CAT_MAP)
log('normalize_category', 'fixed spelling/casing and merged Apparel into Merch', (before_cat != df['category']).sum())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[normalize_category] fixed spelling/casing and merged Apparel into Merch (132 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [9]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].nunique() == 3
assert df['category'].nunique() == 3
assert 'rain poncho' not in df['item'].values
assert 'Apparel' not in df['category'].values
# TODO: assert something about item
# TODO: assert something about category
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:
df['revenue'] = df['qty'] * df['price']

revenue_by_category = (
                      df.groupby('category')['revenue']
                      .sum()
                      .sort_values(ascending=False)
                      .round(2)
                      )

print(revenue_by_category)

overall_revenue = df['revenue'].sum().round(2)
print('Overall Revenue : $',overall_revenue)





category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Overall Revenue : $ 4740.0


**What I would tell the vendor:I would tell teh vendors to emphasize stocking up on food since it is the highest revenue category ($1656)

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop_duplicates,removed exact duplicate rows,15
1,clean price,converted to float,300
2,clean qty,dropped 25 rows with missing or negative qty,25
3,canonicalize_item,turned 6 spellings into 3 items via ITEM_MAP,126
4,normalize_category,fixed spelling/casing and merged Apparel into ...,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.



**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a. The first step (removing duplicates) changed the revenue total the most (changed it by 258.0). The revenue before this step was 4852.5 and the revenue after was 4594.5. The only other TODO that changed the revenue total was removing missing or negatives, but this step only changed the total revenue by 145.50.

b. I decided to put merge the Apparel category title with Merch because I considered them synonyms for hte same thing. A reasonable person could have kept Apparel as a separate category. This would not have affected the total revenue, however, it would have made the Merch category's revenue sum lower.